#  코드 설명
- 성능 올리는 피쳐만 자동으로 selection 버전
- ++ 공행성쌍 판단에 가중치를 두어 공행성을 놓치지 않게 함
- +++ 피쳐 추가함
- ++++기존 베이스라인에서 모델 튜닝 추가
- +++++ OverSampling
- ++++++ 공행성쌍 판단 로직 추가(3필터)
- +++++++ 기존 공행성 판단 코드에서 임계값만 0.325->0.35
- dropout 0.04으로 설정
$$ <BEST> 0.4058459231 oversampling 제거


## 1. Import

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from catboost import CatBoostRegressor
from tqdm import tqdm
from datetime import datetime
from xgboost import XGBRegressor
import random
import os

## 2. 데이터 전처리

In [2]:
# Seed 고정
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# 📁 데이터 경로 한 곳에서 관리
PATH = "/Users/jab/Desktop/DACON/data/train.csv"

train = pd.read_csv(PATH)

- type 컬럼에는 '1'만 있음(컬럼 삭제)
- 결측값 없음
- 컬럼 분포 확인하기
- item_id는 저마다 교유한 hs4를 지님
- 이상치도 파악해보기
- 중량과 수량 모두 0인 값은 635개
- 전월/전년 대비 성장률 피쳐 생성
- 단가 피쳐 생성


================================================================================================

value를 공행성 파악 시에만 scaling, 학습 및 예측에선 X
- 유의미한 성능 변화 없음

In [3]:
# hs4 앞 두 자리 추출
train['hs2'] = train['hs4'].astype(str).str[:2]

# hs2 종류가 몇 개인지 출력
num_hs2 = train['hs2'].nunique()

print("hs4 앞 2자리(hs2) 개수:", num_hs2)

hs4 앞 2자리(hs2) 개수: 39


================================================================================================

In [4]:
# year, month, item_id 기준으로 value 합산 (seq만 다르다면 value 합산)
monthly = (
    train
    .groupby(["item_id", "year", "month"], as_index=False)["value"]
    .sum()
)

# year, month를 하나의 키(ym)로 묶기
monthly["ym"] = pd.to_datetime(
    monthly["year"].astype(str) + "-" + monthly["month"].astype(str).str.zfill(2)
)

# item_id × ym 피벗 (월별 총 무역량 매트릭스 생성)
pivot = (
    monthly
    .pivot(index="item_id", columns="ym", values="value")
    .fillna(0.0)
)

pivot

ym,2022-01-01,2022-02-01,2022-03-01,2022-04-01,2022-05-01,2022-06-01,2022-07-01,2022-08-01,2022-09-01,2022-10-01,...,2024-10-01,2024-11-01,2024-12-01,2025-01-01,2025-02-01,2025-03-01,2025-04-01,2025-05-01,2025-06-01,2025-07-01
item_id,,,,,,,,,,,,,,,,,,,,,
AANGBULD,14276.0,52347.0,53549.0,0.0,26997.0,84489.0,0.0,0.0,0.0,0.0,...,428725.0,144248.0,26507.0,25691.0,25805.0,0.0,38441.0,0.0,441275.0,533478.0
AHMDUILJ,242705.0,120847.0,197317.0,126142.0,71730.0,149138.0,186617.0,169995.0,140547.0,89292.0,...,123085.0,143451.0,78649.0,125098.0,80404.0,157401.0,115509.0,127473.0,89479.0,101317.0
ANWUJOKX,0.0,0.0,0.0,63580.0,81670.0,26424.0,8470.0,0.0,0.0,80475.0,...,0.0,0.0,0.0,27980.0,0.0,0.0,0.0,0.0,0.0,0.0
APQGTRMF,383999.0,512813.0,217064.0,470398.0,539873.0,582317.0,759980.0,216019.0,537693.0,205326.0,...,683581.0,2147.0,0.0,25013.0,77.0,20741.0,2403.0,3543.0,32430.0,40608.0
ATLDMDBO,143097177.0,103568323.0,118403737.0,121873741.0,115024617.0,65716075.0,146216818.0,97552978.0,72341427.0,87454167.0,...,60276050.0,30160198.0,42613728.0,64451013.0,38667429.0,29354408.0,42450439.0,37136720.0,32181798.0,57090235.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YSYHGLQK,0.0,543.0,766.0,1108.0,859.0,1426.0,2413.0,638.0,0.0,1199.0,...,188.0,541.0,696.0,8710.0,3175.0,2624.0,0.0,182.0,2128.0,10651.0
ZCELVYQU,373859.0,59900.0,31158.0,594407.0,648232.0,496737.0,210179.0,0.0,70748.0,15512.0,...,0.0,609803.0,23712.0,654630.0,4496.0,1177300.0,1187539.0,26434.0,115631.0,270262.0
ZGJXVMNI,1154724.0,1337622.0,1662893.0,1561647.0,1603223.0,1641942.0,1815161.0,1546959.0,1536799.0,1496906.0,...,3168505.0,3059865.0,1579976.0,1413293.0,3038078.0,2915914.0,3565526.0,3020051.0,2412781.0,2458481.0


In [5]:
import numpy as np

# 1. 각 행(Item)별로 0이 아닌 값의 개수 세기
nonzero_counts = (pivot != 0).sum(axis=1)

# 2. 개수가 38개 이상인 행의 수 계산
survivors = (nonzero_counts >= 38).sum()

print(f"전체 아이템 수: {len(pivot)}")
print(f"데이터가 38개 이상인 아이템 수: {survivors}")
print(f"생존율: {survivors / len(pivot) * 100:.2f}%")

# (선택사항) 어떤 아이템들이 살아남았는지 보고 싶다면:
# survived_items = nonzero_counts[nonzero_counts >= 38].index.tolist()
# print(survived_items[:10]) # 상위 10개만 출력

전체 아이템 수: 100
데이터가 38개 이상인 아이템 수: 78
생존율: 78.00%


## 3. 공행성쌍 탐색
- 각 (A, B) 쌍에 대해 lag = 1 ~ max_lag까지 Pearson 상관계수 계산
- 절댓값이 가장 큰 상관계수와 lag를 선택
- |corr| >= corr_threshold이면 A→B 공행성 있다고 판단

In [6]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# ==========================================
# 1. 계산 함수들
# ==========================================
def weighted_mean(x, w):
    return np.sum(x * w) / (np.sum(w) + 1e-9)

def weighted_cov(x, y, w):
    mean_x = weighted_mean(x, w)
    mean_y = weighted_mean(y, w)
    return np.sum(w * (x - mean_x) * (y - mean_y)) / (np.sum(w) + 1e-9)

def weighted_corr(x, y, w):
    if np.sum(w) < 1e-9:
        return 0.0
    cov_xy = weighted_cov(x, y, w)
    cov_xx = weighted_cov(x, x, w)
    cov_yy = weighted_cov(y, y, w)
    if cov_xx <= 0 or cov_yy <= 0:
        return 0.0
    return cov_xy / (np.sqrt(cov_xx * cov_yy) + 1e-9)

# ==========================================
# 2. 최종 공행성 탐색 (corr_threshold=0.35 + 분포 기반 약한 후처리)
# ==========================================
def find_comovement_pairs(
    pivot,
    max_lag=6,
    min_nonzero=12,
    corr_threshold=0.35,
    drop_frac=0.04
):
    """
    Basic Version + Filter Tuning
    - corr_threshold = 0.35 (final_score 기준, 그대로 유지)
    - 1차 필터: Jaccard >= 0.30, valid_cnt >= 12
    - 2차 필터(후처리): jaccard / reliability 분포 기준 하위 drop_frac 만큼 제거
      → FP 위주로 정리, TP 손실은 최소화하는 약한 필터
    """

    items = pivot.index.to_list()
    months = pivot.columns.to_list()
    n_months = len(months)

    results = []
    
    print(
        f"🔎 공행성 탐색 (1차: Jaccard>=0.30, valid_cnt>=12, |score|>={corr_threshold})..."
    )
    
    for i, leader in tqdm(enumerate(items), total=len(items)):
        x_full = pivot.loc[leader].values.astype(float)
        if np.count_nonzero(x_full) < min_nonzero:
            continue

        for follower in items:
            if follower == leader:
                continue

            y_full = pivot.loc[follower].values.astype(float)
            if np.count_nonzero(y_full) < min_nonzero:
                continue

            best_lag = None
            best_score = 0.0
            best_metric_corr = 0.0
            best_jaccard = 0.0
            best_rel = 0.0

            for lag in range(1, max_lag + 1):
                if n_months <= lag:
                    continue

                x_slice = x_full[:-lag]
                y_slice = y_full[lag:]
                
                # 1) Timing: Jaccard (기본 컷 0.30)
                x_bool = x_slice > 0
                y_bool = y_slice > 0
                intersection = np.logical_and(x_bool, y_bool).sum()
                union = np.logical_or(x_bool, y_bool).sum()
                jaccard = intersection / (union + 1e-9)

                if jaccard < 0.30:
                    continue

                # 2) Weighted corr
                w = np.log1p(x_slice) * np.log1p(y_slice)
                if np.sum(w) < 1e-9:
                    continue

                w_corr = weighted_corr(x_slice, y_slice, w)

                # 3) Reliability: 같이 의미 있게 거래된 달 수
                valid_cnt = np.count_nonzero(w > 0)
                if valid_cnt < 12:
                    rel = 0.0
                else:
                    rel = np.sqrt(valid_cnt / (n_months - lag))

                # 4) 최종 점수 (이 기준으로 corr_threshold 적용)
                final_score = w_corr * jaccard * rel

                if abs(final_score) > abs(best_score):
                    best_score = final_score
                    best_metric_corr = w_corr
                    best_lag = lag
                    best_jaccard = jaccard
                    best_rel = rel

            # corr_threshold는 final_score에 적용 (유지)
            if best_lag is not None and abs(best_score) >= corr_threshold:
                results.append({
                    "leading_item_id": leader,
                    "following_item_id": follower,
                    "best_lag": best_lag,
                    "max_corr": best_metric_corr,
                    "score": best_score,
                    "jaccard": best_jaccard,
                    "reliability": best_rel,
                })

    pairs = pd.DataFrame(results)
    print(f"✅ 1차 필터까지 통과한 쌍: {len(pairs)}개")

    if pairs.empty:
        print("⚠️ 공행성 쌍이 하나도 없습니다.")
        return pairs

    # ---------------------------------------------------------
    # 🔥 2차 필터: jaccard / reliability 분포 기반 약한 컷
    # ---------------------------------------------------------
    pairs['abs_corr'] = pairs['max_corr'].abs()

    j_cut = pairs['jaccard'].quantile(drop_frac)
    r_cut = pairs['reliability'].quantile(drop_frac)

    print(
        f"✂️ 2차 필터: jaccard < {j_cut:.4f} 또는 "
        f"reliability < {r_cut:.4f} 인 하위 {int(drop_frac*100)}% 쌍 제거..."
    )

    pairs_filtered = pairs[
        (pairs['jaccard'] >= j_cut) &
        (pairs['reliability'] >= r_cut)
    ].copy()

    print(f"🎉 최종 쌍 개수: {len(pairs_filtered)}개 "
          f"(1차 {len(pairs)} → 2차 {len(pairs_filtered)})")

    return pairs_filtered

/Users/jab/miniforge3/envs/ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# ==========================================
# 3. 실행 및 결과 확인
# ==========================================
# df_train_model을 만들기 위한 pivot이 있다고 가정
# pairs = find_comovement_pairs(pivot) 
# 위 코드는 pivot 변수가 정의된 상태에서 실행하세요.

# (예시 실행)
pairs = find_comovement_pairs(pivot)
print("탐색된 공행성쌍 수:", len(pairs))

# 정렬해서 상위 쌍 확인
if not pairs.empty:
    pairs = pairs.sort_values(by='score', ascending=False, key=abs)
    display(pairs.head())
else:
    print("⚠️ 탐색된 쌍이 없습니다. Threshold를 낮추거나 min_nonzero를 조절해보세요.")

🔎 공행성 탐색 (1차: Jaccard>=0.30, valid_cnt>=12, |score|>=0.35)...


100%|██████████| 100/100 [00:01<00:00, 54.62it/s]

✅ 1차 필터까지 통과한 쌍: 1973개
✂️ 2차 필터: jaccard < 0.6757 또는 reliability < 0.8216 인 하위 4% 쌍 제거...
🎉 최종 쌍 개수: 1893개 (1차 1973 → 2차 1893)
탐색된 공행성쌍 수: 1893


,leading_item_id,following_item_id,best_lag,max_corr,score,jaccard,reliability,abs_corr
580,FTSVTTSR,LLHREMKS,2,0.917683,0.884315,0.97561,0.98773,0.917683
1843,XUOIQPFL,QVLMOEYE,5,0.852786,0.852786,1.00000,1.00000,0.852786
1188,QRKRBYJL,QVLMOEYE,2,0.849164,0.849164,1.00000,1.00000,0.849164
78,ATLDMDBO,QRKRBYJL,1,0.823877,0.823877,1.00000,1.00000,0.823877
925,LRVGFDFM,QVLMOEYE,4,0.794116,0.794116,1.00000,1.00000,0.794116


# 학습

## 4. 학습 데이터 생성
- 시계열 데이터 안에서 '한 달 뒤 총 무역량(value)을 맞추는 문제'로 self-supervised 학습
- 탐색된 모든 공행성쌍 (A,B)에 대해 월 t마다 학습 샘플 생성
- input X:
1) B_t (현재 총 무역량(value))
2) B_{t-1} (직전 달 총 무역량(value))
3) A_{t-lag} (lag 반영된 총 무역량(value))
4) max_corr, best_lag (관계 특성)
- target y:
1) B_{t+1} (다음 달 총 무역량(value))
- 이러한 모든 샘플을 합쳐 LinearRegression 회귀 모델을 학습

In [8]:
def safe_corr(x, y):
    if np.std(x) == 0 or np.std(y) == 0:
        return 0.0
    return float(np.corrcoef(x, y)[0, 1])

In [9]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

def build_training_data(pivot, pairs, train):
    months = pivot.columns.to_list()
    n_months = len(months)

    # 연도, 월 숫자 배열
    years = np.array([dt.year for dt in months])
    months_num = np.array([dt.month for dt in months])
    
    # ---------------------------------------------------------
    # 🧱 0. train 기반 품목 메타 정보(정적인 피처) 계산
    # ---------------------------------------------------------
    # item_id별 hs4, weight/quantity/value 평균·합계
    item_meta = (
        train
        .groupby("item_id", as_index=False)
        .agg(
            hs4=("hs4", "first"),
            weight_mean=("weight", "mean"),
            weight_sum=("weight", "sum"),
            qty_mean=("quantity", "mean"),
            qty_sum=("quantity", "sum"),
            value_mean=("value", "mean")
        )
    )
    item_meta["hs2"] = (item_meta["hs4"] // 100).astype(int)
    item_meta = item_meta.set_index("item_id")

    # dict 형태로 빠르게 lookup 하기 위함
    meta_dict = item_meta.to_dict(orient="index")

    # meta 정보가 없는 경우에 대비한 기본값
    meta_default = {
        "hs4": 0,
        "hs2": 0,
        "weight_mean": 0.0,
        "weight_sum": 0.0,
        "qty_mean": 0.0,
        "qty_sum": 0.0,
        "value_mean": 0.0,
    }

    # ---------------------------------------------------------
    # 📊 1. 연도별 & 여름(Summer) 통계 사전 계산
    # ---------------------------------------------------------
    print("📊 연도별/여름 통계 사전 계산 중...")
    
    yearly_stats = {}   # (item_id, year) -> (mean, sum)
    summer_stats_by_year = {} # (item_id, year) -> {mean, sum, std, ...}

    unique_years = np.unique(years)
    items_list = pivot.index.to_list()
    pivot_values = pivot.values
    
    SUMMER_MONTHS = [6, 7, 8]

    for y in unique_years:
        # 1) 연도 마스크
        mask_y = (years == y)
        if not np.any(mask_y): 
            continue
        
        data_y = pivot_values[:, mask_y]
        
        # --- A. 연도별 통계 (Yearly) ---
        with np.errstate(divide='ignore', invalid='ignore'):
            m_y = np.nanmean(data_y, axis=1)
            s_y = np.nansum(data_y, axis=1)
        
        for i, item in enumerate(items_list):
            yearly_stats[(item, y)] = (
                0.0 if np.isnan(m_y[i]) else m_y[i],
                0.0 if np.isnan(s_y[i]) else s_y[i]
            )

        # --- B. 여름 통계 (Summer Analysis) ---
        mask_summer = mask_y & np.isin(months_num, SUMMER_MONTHS)
        
        if np.any(mask_summer):
            data_s = pivot_values[:, mask_summer]
            
            with np.errstate(divide='ignore', invalid='ignore'):
                feat_mean = np.nanmean(data_s, axis=1)
                feat_sum  = np.nansum(data_s, axis=1)
                feat_std  = np.nanstd(data_s, axis=1)
                feat_max  = np.nanmax(data_s, axis=1)
                feat_min  = np.nanmin(data_s, axis=1)
                feat_gap  = feat_max - feat_min
            
            for i, item in enumerate(items_list):
                stats = {
                    'mean': 0.0 if np.isnan(feat_mean[i]) else feat_mean[i],
                    'sum':  0.0 if np.isnan(feat_sum[i])  else feat_sum[i],
                    'std':  0.0 if np.isnan(feat_std[i])  else feat_std[i],
                    'max':  0.0 if np.isnan(feat_max[i])  else feat_max[i],
                    'min':  0.0 if np.isnan(feat_min[i])  else feat_min[i],
                    'gap':  0.0 if np.isnan(feat_gap[i])  else feat_gap[i]
                }
                summer_stats_by_year[(item, y)] = stats

    print("✅ 통계 계산 완료. 데이터셋 구축 시작...")

    rows = []
    
    # [설정] 고정적으로 추출할 여름 연도 목록
    TARGET_SUMMER_YEARS = [2022, 2023, 2024, 2025]

    # ---------------------------------------------------------
    # 🚀 2. 학습 데이터 구축 Loop
    # ---------------------------------------------------------
    for row in pairs.itertuples(index=False):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)
        corr = float(row.max_corr)

        if leader not in pivot.index or follower not in pivot.index:
            continue

        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)

        # ----- 쌍(A,B)에 대한 train 기반 정적 메타 피처 -----
        meta_a = meta_dict.get(leader, meta_default)
        meta_b = meta_dict.get(follower, meta_default)

        a_hs4 = int(meta_a["hs4"])
        b_hs4 = int(meta_b["hs4"])
        a_hs2 = int(meta_a["hs2"])
        b_hs2 = int(meta_b["hs2"])
        same_hs2 = 1 if a_hs2 == b_hs2 and a_hs2 != 0 else 0

        pair_static_feats = {
            "a_hs4": a_hs4,
            "b_hs4": b_hs4,
            "a_hs2": a_hs2,
            "b_hs2": b_hs2,
            "same_hs2": same_hs2,
            "a_weight_mean_item": float(meta_a["weight_mean"]),
            "b_weight_mean_item": float(meta_b["weight_mean"]),
            "a_weight_sum_item": float(meta_a["weight_sum"]),
            "b_weight_sum_item": float(meta_b["weight_sum"]),
            "a_qty_mean_item": float(meta_a["qty_mean"]),
            "b_qty_mean_item": float(meta_b["qty_mean"]),
            "a_qty_sum_item": float(meta_a["qty_sum"]),
            "b_qty_sum_item": float(meta_b["qty_sum"]),
            "a_value_mean_item": float(meta_a["value_mean"]),
            "b_value_mean_item": float(meta_b["value_mean"]),
        }

        # ratio 형태 피처도 추가 (0 division 방지)
        pair_static_feats["weight_mean_ratio_a_over_b"] = (
            pair_static_feats["a_weight_mean_item"] /
            (pair_static_feats["b_weight_mean_item"] + 1e-8)
        )
        pair_static_feats["qty_mean_ratio_a_over_b"] = (
            pair_static_feats["a_qty_mean_item"] /
            (pair_static_feats["b_qty_mean_item"] + 1e-8)
        )
        pair_static_feats["value_mean_ratio_a_over_b"] = (
            pair_static_feats["a_value_mean_item"] /
            (pair_static_feats["b_value_mean_item"] + 1e-8)
        )

        # [Rolling Pre-calc]
        b_rolling_3_lag1 = pd.Series(b_series).rolling(3).mean().shift(1).fillna(0.0).values
        b_rolling_std_6 = pd.Series(b_series).rolling(6).std().fillna(0.0).values
        a_rolling_3 = pd.Series(a_series).rolling(3).mean().fillna(0.0).values

        start_t = max(lag + 2, 6)

        for t in range(start_t, n_months - 1):
            b_t_plus_1 = b_series[t + 1]
            year_t = years[t]
            month_t = months_num[t]

            # --- [Basic Lags] ---
            b_t = b_series[t]
            b_t_1 = b_series[t - 1]
            b_t_2 = b_series[t - 2]
            b_t_3 = b_series[t - 3]

            a_t_lag = a_series[t - lag]
            a_t_lag_1 = a_series[t - lag - 1] 
            
            # --- [Advanced Derived] ---
            b_diff = b_t - b_t_1
            a_diff_lag = a_t_lag - a_t_lag_1
            a_roll_mean_3_lag = a_rolling_3[t - lag]
            
            ratio_a_b = a_t_lag / (b_t + 1e-8)
            ratio_diff = a_diff_lag / (b_diff + 1e-8) if abs(b_diff) > 1e-9 else 0.0

            # --- [Historical Stats (Dynamic)] ---
            prev_year = year_t - 1
            b_yr_mean, b_yr_sum = yearly_stats.get((follower, prev_year), (0.0, 0.0))
            a_yr_mean, a_yr_sum = yearly_stats.get((leader, prev_year), (0.0, 0.0))

            # --- [Feature Dict 초기화] ---
            row_dict = {
                "b_t": b_t, "b_t_1": b_t_1, "b_t_2": b_t_2, "b_t_3": b_t_3,
                "b_diff": b_diff, "b_roll_mean_3_lag1": b_rolling_3_lag1[t], "b_std_6": b_rolling_std_6[t],
                "a_t_lag": a_t_lag, "a_diff_lag": a_diff_lag, "a_roll_mean_3_lag": a_roll_mean_3_lag,
                "max_corr": corr, "best_lag": float(lag), "ratio_a_b": ratio_a_b, "ratio_diff": ratio_diff,
                "month_t": float(month_t),
                "b_year_mean": b_yr_mean, "b_year_sum": b_yr_sum,
                "a_year_mean": a_yr_mean, "a_year_sum": a_yr_sum,
                "target": b_t_plus_1
            }

            # ----- 쌍(A,B) 정적 메타 피처 추가 -----
            row_dict.update(pair_static_feats)

            # --- [Fixed Summer Stats: 22, 23, 24, 25] ---
            for target_y in TARGET_SUMMER_YEARS:
                is_future = (target_y > year_t)
                
                if is_future:
                    stats_a = {'mean':0,'sum':0,'std':0,'max':0,'min':0,'gap':0}
                    stats_b = {'mean':0,'sum':0,'std':0,'max':0,'min':0,'gap':0}
                else:
                    if year_t > target_y:
                        stats_b = summer_stats_by_year.get(
                            (follower, target_y), 
                            {'mean':0,'sum':0,'std':0,'max':0,'min':0,'gap':0}
                        )
                        stats_a = summer_stats_by_year.get(
                            (leader, target_y), 
                            {'mean':0,'sum':0,'std':0,'max':0,'min':0,'gap':0}
                        )
                    elif year_t == target_y and year_t == 2025:
                        stats_b = summer_stats_by_year.get(
                            (follower, target_y), 
                            {'mean':0,'sum':0,'std':0,'max':0,'min':0,'gap':0}
                        )
                        stats_a = summer_stats_by_year.get(
                            (leader, target_y), 
                            {'mean':0,'sum':0,'std':0,'max':0,'min':0,'gap':0}
                        )
                    else:
                        stats_b = {'mean':0,'sum':0,'std':0,'max':0,'min':0,'gap':0}
                        stats_a = {'mean':0,'sum':0,'std':0,'max':0,'min':0,'gap':0}

                suffix = str(target_y)[-2:] # 22, 23, 24, 25
                for k in ['mean', 'sum', 'std', 'max', 'min', 'gap']:
                    row_dict[f"b_summer_{suffix}_{k}"] = stats_b[k]
                    row_dict[f"a_summer_{suffix}_{k}"] = stats_a[k]

            rows.append(row_dict)

    df_train = pd.DataFrame(rows).fillna(0.0)
    
    return df_train

#### 성능 향상 여부에 따른 Feature Selection

In [11]:
# ==========================================
# 📌 학습 데이터 생성 + feature_cols만 유지
# ==========================================

df_train_model = build_training_data(pivot, pairs, train)

print("원본 학습 데이터 shape:", df_train_model.shape)

# 🔥 feature_cols + target 만 남기고 나머지 삭제
feature_cols = ['b_t', 'b_t_1', 'a_t_lag', 'max_corr', 'best_lag', 'b_t_2', 'b_t_3', 'b_roll_mean_3_lag1', 'b_std_6', 'month_t', 'b_value_mean_item', 'b_diff', 'b_hs4', 'a_hs2', 'b_weight_mean_item', 'a_weight_sum_item', 'b_weight_sum_item', 'b_summer_22_std', 'b_summer_23_mean', 'b_summer_23_std', 'a_summer_23_std', 'b_summer_23_max', 'a_summer_23_max', 'a_summer_23_min']
use_cols = feature_cols + ["target"]
df_train_model = df_train_model[use_cols]

print("필터링된 학습 데이터 shape:", df_train_model.shape)

df_train_model.head()

📊 연도별/여름 통계 사전 계산 중...
✅ 통계 계산 완료. 데이터셋 구축 시작...
원본 학습 데이터 shape: (67110, 86)
필터링된 학습 데이터 shape: (67110, 25)


,b_t,b_t_1,a_t_lag,max_corr,best_lag,b_t_2,b_t_3,b_roll_mean_3_lag1,b_std_6,month_t,...,a_weight_sum_item,b_weight_sum_item,b_summer_22_std,b_summer_23_mean,b_summer_23_std,a_summer_23_std,b_summer_23_max,a_summer_23_max,a_summer_23_min,target
0,24506.0,49367.0,241787.0,0.917683,2.0,24723.0,67977.0,47355.666667,37864.010906,7.0,...,14018605.0,11236613.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,113568.0
1,113568.0,24506.0,191752.0,0.917683,2.0,49367.0,24723.0,32865.333333,33132.671149,8.0,...,14018605.0,11236613.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20109.0
2,20109.0,113568.0,46240.0,0.917683,2.0,24506.0,49367.0,62480.333333,36185.068941,9.0,...,14018605.0,11236613.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,23918.0
3,23918.0,20109.0,124546.0,0.917683,2.0,113568.0,24506.0,52727.666667,36287.830659,10.0,...,14018605.0,11236613.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,38770.0
4,38770.0,23918.0,24568.0,0.917683,2.0,20109.0,113568.0,52531.666667,35336.840304,11.0,...,14018605.0,11236613.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,33891.0


### MODEL

In [14]:
# ==================================================================================
# 🏆 4-Model Battle: CatBoost vs LGBM vs XGB vs RF
# 📊 Strategy: Optuna Tuning x 5-Fold CV -> Performance Ranking
# 🛠️ Fix: LGBM (callbacks), XGB (constructor early_stopping) API 호환성 해결
# ==================================================================================

import numpy as np
import pandas as pd
import optuna
import matplotlib.pyplot as plt
import seaborn as sns

# 모델 라이브러리
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor

# 평가 및 유틸리티
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

# Optuna 로그 레벨 조정
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==========================================
# 0) Helper Functions (NMAE)
# ==========================================
def nmae_score(y_true, y_pred):
    error_ratio = np.abs(y_true - y_pred) / (np.abs(y_true) + 1e-8)
    capped_error = np.minimum(1, error_ratio)
    return np.mean(capped_error)

# ==========================================
# 1) 데이터 준비
# ==========================================
# df_train_model이 이미 메모리에 있다고 가정
feature_cols = df_train_model.columns.drop('target').tolist()
train_X = df_train_model[feature_cols].values
train_y = df_train_model["target"].values

print(f"✅ 데이터 준비 완료: {train_X.shape}")
print("="*60)

# ==========================================
# 2) 모델별 Optuna Objective 정의
# ==========================================
kf = KFold(n_splits=5, shuffle=True, random_state=42)
N_TRIALS = 15  # 튜닝 횟수

def get_objective(model_name):
    def objective(trial):
        # ------------------------------------
        # A. CatBoost
        # ------------------------------------
        if model_name == 'CatBoost':
            params = {
                "iterations": 1500,
                "depth": trial.suggest_int("depth", 4, 10),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
                "subsample": trial.suggest_float("subsample", 0.6, 0.95),
                "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.6, 0.95),
                "random_strength": trial.suggest_float("random_strength", 1, 5),
                "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 50),
                "loss_function": "MAE",
                "eval_metric": "MAE",
                "verbose": 0,
                "random_seed": 42,
                "allow_writing_files": False
            }
            model = CatBoostRegressor(**params)

        # ------------------------------------
        # B. LightGBM
        # ------------------------------------
        elif model_name == 'LGBM':
            params = {
                "n_estimators": 1500,
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                "num_leaves": trial.suggest_int("num_leaves", 16, 256),
                "max_depth": trial.suggest_int("max_depth", 4, 12),
                "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
                "subsample": trial.suggest_float("subsample", 0.6, 0.95),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.95),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
                "objective": "mae",
                "metric": "mae",
                "verbosity": -1,
                "random_state": 42
            }
            model = LGBMRegressor(**params)

        # ------------------------------------
        # C. XGBoost (🔥 Fix: early_stopping_rounds를 생성자에 넣음)
        # ------------------------------------
        elif model_name == 'XGB':
            params = {
                "n_estimators": 1500,
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                "max_depth": trial.suggest_int("max_depth", 4, 10),
                "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
                "subsample": trial.suggest_float("subsample", 0.6, 0.95),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.95),
                "gamma": trial.suggest_float("gamma", 0, 5),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
                "objective": "reg:absoluteerror",
                "eval_metric": "mae",
                "verbosity": 0,
                "random_state": 42,
                "n_jobs": -1,
                "early_stopping_rounds": 30  # 🔥 생성자에 추가
            }
            model = XGBRegressor(**params)

        # ------------------------------------
        # D. Random Forest
        # ------------------------------------
        elif model_name == 'RF':
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 100, 300),
                "max_depth": trial.suggest_int("max_depth", 6, 16),
                "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
                "max_features": trial.suggest_float("max_features", 0.5, 1.0),
                "criterion": "absolute_error", 
                "random_state": 42,
                "n_jobs": -1
            }
            model = RandomForestRegressor(**params)

        # --- 5-Fold Cross Validation ---
        scores = []
        for tr_idx, val_idx in kf.split(train_X):
            X_tr, X_val = train_X[tr_idx], train_X[val_idx]
            y_tr, y_val = train_y[tr_idx], train_y[val_idx]

            # 🔥 [수정됨] 모델별 학습 방식 분기
            if model_name == 'LGBM':
                # LightGBM: callbacks 사용
                model.fit(
                    X_tr, y_tr,
                    eval_set=[(X_val, y_val)],
                    eval_metric='mae',
                    callbacks=[
                        early_stopping(stopping_rounds=30, verbose=False), 
                        log_evaluation(0)
                    ]
                )
            elif model_name == 'XGB':
                # XGBoost: 생성자에서 설정했으므로 fit에서는 제거
                model.fit(
                    X_tr, y_tr,
                    eval_set=[(X_val, y_val)],
                    verbose=False
                )
            elif model_name == 'CatBoost':
                # CatBoost: fit 인자 사용 유지
                model.fit(
                    X_tr, y_tr,
                    eval_set=[(X_val, y_val)],
                    early_stopping_rounds=30,
                    verbose=False
                )
            else:
                # RF (Early Stopping 없음)
                model.fit(X_tr, y_tr)

            # 예측 및 평가
            pred = model.predict(X_val)
            pred = np.maximum(0, pred)
            
            score = nmae_score(y_val, pred)
            scores.append(score)

        return np.mean(scores)
    
    return objective

# ==========================================
# 3) 모델 비교 및 튜닝 실행
# ==========================================
models_list = ['CatBoost', 'LGBM', 'XGB', 'RF']
results = []

print(f"🚀 4-Model Comparison Start (Trials per model: {N_TRIALS})")
print("="*60)

final_models_dict = {}

for m_name in models_list:
    print(f"🔹 Tuning {m_name}...")
    
    study = optuna.create_study(direction="minimize")
    study.optimize(get_objective(m_name), n_trials=N_TRIALS)
    
    best_score = study.best_value
    best_params = study.best_params
    
    print(f"   -> Best NMAE: {best_score:.6f}")
    
    results.append({
        "Model": m_name,
        "Best_NMAE": best_score,
        "Best_Params": best_params
    })
    
    # 최적 파라미터로 최종 모델 객체 생성 (재학습 준비)
    if m_name == 'CatBoost':
        best_params.update({"iterations": 2000, "loss_function": "MAE", "eval_metric": "MAE", "verbose": 0, "allow_writing_files": False, "random_seed": 42})
        final_model = CatBoostRegressor(**best_params)
    elif m_name == 'LGBM':
        best_params.update({"n_estimators": 2000, "objective": "mae", "metric": "mae", "verbosity": -1, "random_state": 42})
        final_model = LGBMRegressor(**best_params)
    elif m_name == 'XGB':
        # 최종 모델에서도 생성자에 early stopping 포함 (필요 시)
        best_params.update({"n_estimators": 2000, "objective": "reg:absoluteerror", "eval_metric": "mae", "verbosity": 0, "random_state": 42, "n_jobs": -1, "early_stopping_rounds": 50})
        final_model = XGBRegressor(**best_params)
    elif m_name == 'RF':
        best_params.update({"random_state": 42, "n_jobs": -1, "criterion": "absolute_error"})
        final_model = RandomForestRegressor(**best_params)
        
    final_models_dict[m_name] = final_model

print("="*60)
print("🏁 Tuning Completed!")

# ==========================================
# 4) 최종 순위 및 리포트
# ==========================================
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by="Best_NMAE", ascending=True).reset_index(drop=True)
df_results["Rank"] = df_results.index + 1

print("\n📊 [Model Performance Leaderboard]")
print(df_results[["Rank", "Model", "Best_NMAE"]])

# 시각화 
plt.figure(figsize=(10, 6))
sns.barplot(data=df_results, x="Model", y="Best_NMAE", palette="viridis")
plt.title("4-Model Comparison (Lower NMAE is Better)")
plt.ylim(df_results["Best_NMAE"].min() * 0.9, df_results["Best_NMAE"].max() * 1.1)
plt.ylabel("NMAE Score")
plt.show()

# ==========================================
# 5) 가장 성능 좋은 모델로 'model' 객체 설정
# ==========================================
best_model_name = df_results.iloc[0]["Model"]
print(f"\n🏆 The Winner is: {best_model_name}")

final_best_model = final_models_dict[best_model_name]

# 전체 데이터 학습
if best_model_name == 'LGBM':
    # 전체 학습 시에도 callbacks 필요할 수 있음 (검증셋 없으면 경고 뜨지만 학습은 됨)
    # 여기서는 안전하게 검증셋 없이 전체 학습 수행
    final_best_model.fit(train_X, train_y)
elif best_model_name == 'XGB':
    # XGB는 검증셋 없으면 early stopping 무시됨 (정상)
    final_best_model.fit(train_X, train_y, verbose=False)
elif best_model_name == 'CatBoost':
    final_best_model.fit(train_X, train_y, verbose=False)
else:
    final_best_model.fit(train_X, train_y)

print(f"✅ Final Model ({best_model_name}) retrained on full data.")
model = final_best_model

✅ 데이터 준비 완료: (67110, 24)
🚀 4-Model Comparison Start (Trials per model: 15)
🔹 Tuning CatBoost...
   -> Best NMAE: 0.081970
🔹 Tuning LGBM...


/Users/jab/miniforge3/envs/ml/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/jab/miniforge3/envs/ml/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/jab/miniforge3/envs/ml/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/jab/miniforge3/envs/ml/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/jab/miniforge3/envs/ml/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fit

   -> Best NMAE: 0.046712
🔹 Tuning XGB...
   -> Best NMAE: 0.025939
🔹 Tuning RF...


[W 2025-11-28 08:04:48,697] Trial 0 failed with parameters: {'n_estimators': 275, 'max_depth': 16, 'min_samples_split': 2, 'min_samples_leaf': 3, 'max_features': 0.9554156005158478} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/Users/jab/miniforge3/envs/ml/lib/python3.11/site-packages/optuna/study/_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/var/folders/4y/lcn2sqw52kj_mbxs01807spc0000gn/T/ipykernel_7294/2638676004.py", line 169, in objective
    model.fit(X_tr, y_tr)
  File "/Users/jab/miniforge3/envs/ml/lib/python3.11/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/jab/miniforge3/envs/ml/lib/python3.11/site-packages/sklearn/ensemble/_forest.py", line 486, in fit
    trees = Parallel(
            ^^^^^^^^^
  File "/Users/jab/miniforge3/envs/ml/lib

KeyboardInterrupt: 

## 5. 회귀 모델 추론 및 제출(submission) 파일 생성
- 탐색된 공행성 쌍에 대해 후행 품목(following_item_id)에 대한 2025년 8월 총 무역량(value) 예측

In [ ]:
def predict(pivot, pairs, reg, train):
    months = pivot.columns.to_list()
    n_months = len(months)
    years = np.array([dt.year for dt in months])
    months_num = np.array([dt.month for dt in months])

    # ---------------------------------------------------------
    # 🧱 0. train 기반 품목 메타 정보(정적인 피처) 계산
    # ---------------------------------------------------------
    item_meta = (
        train
        .groupby("item_id", as_index=False)
        .agg(
            hs4=("hs4", "first"),
            weight_mean=("weight", "mean"),
            weight_sum=("weight", "sum"),
            qty_mean=("quantity", "mean"),
            qty_sum=("quantity", "sum"),
            value_mean=("value", "mean")
        )
    )
    item_meta["hs2"] = (item_meta["hs4"] // 100).astype(int)
    item_meta = item_meta.set_index("item_id")

    meta_dict = item_meta.to_dict(orient="index")
    meta_default = {
        "hs4": 0,
        "hs2": 0,
        "weight_mean": 0.0,
        "weight_sum": 0.0,
        "qty_mean": 0.0,
        "qty_sum": 0.0,
        "value_mean": 0.0,
    }

    # ---------------------------------------------------------
    # 📊 연도별 & 여름 통계
    # ---------------------------------------------------------
    print("📊 예측용 연도별/여름 통계 사전 계산 중...")
    
    yearly_stats = {}
    summer_stats_by_year = {}

    unique_years = np.unique(years)
    items_list = pivot.index.to_list()
    pivot_values = pivot.values
    SUMMER_MONTHS = [6, 7, 8]

    for y in unique_years:
        mask_y = (years == y)
        if not np.any(mask_y): continue

        data_y = pivot_values[:, mask_y]

        with np.errstate(divide='ignore', invalid='ignore'):
            m_y = np.nanmean(data_y, axis=1)
            s_y = np.nansum(data_y, axis=1)

        for i, item in enumerate(items_list):
            yearly_stats[(item, y)] = (
                0.0 if np.isnan(m_y[i]) else m_y[i],
                0.0 if np.isnan(s_y[i]) else s_y[i]
            )

        mask_summer = mask_y & np.isin(months_num, SUMMER_MONTHS)
        if np.any(mask_summer):
            data_s = pivot_values[:, mask_summer]
            with np.errstate(divide='ignore', invalid='ignore'):
                feat_mean = np.nanmean(data_s, axis=1)
                feat_sum  = np.nansum(data_s, axis=1)
                feat_std  = np.nanstd(data_s, axis=1)
                feat_max  = np.nanmax(data_s, axis=1)
                feat_min  = np.nanmin(data_s, axis=1)
                feat_gap  = feat_max - feat_min

            for i, item in enumerate(items_list):
                summer_stats_by_year[(item, y)] = {
                    'mean': 0.0 if np.isnan(feat_mean[i]) else feat_mean[i],
                    'sum':  0.0 if np.isnan(feat_sum[i])  else feat_sum[i],
                    'std':  0.0 if np.isnan(feat_std[i])  else feat_std[i],
                    'max':  0.0 if np.isnan(feat_max[i])  else feat_max[i],
                    'min':  0.0 if np.isnan(feat_min[i])  else feat_min[i],
                    'gap':  0.0 if np.isnan(feat_gap[i])  else feat_gap[i]
                }

    print("✅ 예측 데이터 생성 및 추론 시작")

    rows = []
    ids_for_submission = []

    t = n_months - 1
    year_t = years[t]
    month_t = months_num[t]
    TARGET_SUMMER_YEARS = [2022, 2023, 2024, 2025]

    for row in tqdm(pairs.itertuples(index=False), total=len(pairs)):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)
        corr = float(row.max_corr)

        if leader not in pivot.index or follower not in pivot.index: continue
        if t < max(lag + 2, 6): continue

        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)

        meta_a = meta_dict.get(leader, meta_default)
        meta_b = meta_dict.get(follower, meta_default)

        a_hs4 = int(meta_a["hs4"])
        b_hs4 = int(meta_b["hs4"])
        a_hs2 = int(meta_a["hs2"])
        b_hs2 = int(meta_b["hs2"])
        same_hs2 = 1 if a_hs2 == b_hs2 and a_hs2 != 0 else 0

        pair_static_feats = {
            "a_hs4": a_hs4, "b_hs4": b_hs4,
            "a_hs2": a_hs2, "b_hs2": b_hs2,
            "same_hs2": same_hs2,
            "a_weight_mean_item": float(meta_a["weight_mean"]),
            "b_weight_mean_item": float(meta_b["weight_mean"]),
            "a_weight_sum_item": float(meta_a["weight_sum"]),
            "b_weight_sum_item": float(meta_b["weight_sum"]),
            "a_qty_mean_item": float(meta_a["qty_mean"]),
            "b_qty_mean_item": float(meta_b["qty_mean"]),
            "a_qty_sum_item": float(meta_a["qty_sum"]),
            "b_qty_sum_item": float(meta_b["qty_sum"]),
            "a_value_mean_item": float(meta_a["value_mean"]),
            "b_value_mean_item": float(meta_b["value_mean"]),
        }

        pair_static_feats["weight_mean_ratio_a_over_b"] = pair_static_feats["a_weight_mean_item"] / (pair_static_feats["b_weight_mean_item"] + 1e-8)
        pair_static_feats["qty_mean_ratio_a_over_b"] = pair_static_feats["a_qty_mean_item"] / (pair_static_feats["b_qty_mean_item"] + 1e-8)
        pair_static_feats["value_mean_ratio_a_over_b"] = pair_static_feats["a_value_mean_item"] / (pair_static_feats["b_value_mean_item"] + 1e-8)

        # Rolling & lag features
        b_roll_mean_3_lag1 = np.mean(b_series[t-3:t]) if t >= 3 else 0.0
        b_std_6 = np.std(b_series[t-5:t+1]) if t >= 5 else 0.0
        idx_a = t - lag
        a_roll_mean_3_lag = np.mean(a_series[idx_a-2:idx_a+1]) if idx_a >= 2 else 0.0

        b_t = b_series[t]
        b_t_1 = b_series[t-1]
        b_t_2 = b_series[t-2]
        b_t_3 = b_series[t-3]

        a_t_lag = a_series[t-lag]
        a_t_lag_1 = a_series[t-lag-1]

        b_diff = b_t - b_t_1
        a_diff_lag = a_t_lag - a_t_lag_1
        ratio_a_b = a_t_lag / (b_t + 1e-8)
        ratio_diff = a_diff_lag / (b_diff + 1e-8) if abs(b_diff) > 1e-9 else 0.0

        prev_year = year_t - 1
        b_yr_mean, b_yr_sum = yearly_stats.get((follower, prev_year), (0.0, 0.0))
        a_yr_mean, a_yr_sum = yearly_stats.get((leader, prev_year), (0.0, 0.0))

        row_dict = {
            "b_t": b_t, "b_t_1": b_t_1, "b_t_2": b_t_2, "b_t_3": b_t_3,
            "b_diff": b_diff, "b_roll_mean_3_lag1": b_roll_mean_3_lag1, "b_std_6": b_std_6,
            "a_t_lag": a_t_lag, "a_diff_lag": a_diff_lag, "a_roll_mean_3_lag": a_roll_mean_3_lag,
            "max_corr": corr, "best_lag": float(lag), "ratio_a_b": ratio_a_b, "ratio_diff": ratio_diff,
            "month_t": float(month_t),
            "b_year_mean": b_yr_mean, "b_year_sum": b_yr_sum,
            "a_year_mean": a_yr_mean, "a_year_sum": a_yr_sum,
        }

        row_dict.update(pair_static_feats)

        for target_y in TARGET_SUMMER_YEARS:
            stats_b = summer_stats_by_year.get((follower, target_y), {'mean':0,'sum':0,'std':0,'max':0,'min':0,'gap':0})
            stats_a = summer_stats_by_year.get((leader, target_y), {'mean':0,'sum':0,'std':0,'max':0,'min':0,'gap':0})

            suffix = str(target_y)[-2:]
            for k in ['mean', 'sum', 'std', 'max', 'min', 'gap']:
                row_dict[f"b_summer_{suffix}_{k}"] = stats_b[k]
                row_dict[f"a_summer_{suffix}_{k}"] = stats_a[k]

        rows.append(row_dict)
        ids_for_submission.append({"leading_item_id": leader, "following_item_id": follower})

    X_test_df = pd.DataFrame(rows).fillna(0.0)
    try:
        X_test = X_test_df[feature_cols]
    except:
        X_test = X_test_df

    y_pred = reg.predict(X_test)

    df_pred = pd.DataFrame(ids_for_submission)
    df_pred["value"] = np.maximum(0, y_pred).round().astype(int)

    return df_pred

In [ ]:
# ======================================================
# 📁 제출 파일 자동 생성 (날짜+시간+모델명 포함)
# ======================================================
# 현재 시각 문자열 생성
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# 모델명 자동 추출
MODEL = model
model_name = MODEL.__class__.__name__

# 파일명 지정
SUBMISSION_PATH = f"./submission/Model_Compare_{timestamp}.csv"

# SUBMISSION 생성
submission = predict(pivot, pairs, MODEL, train)

# 저장
submission.to_csv(SUBMISSION_PATH, index=False)

print(f"✅ 제출 파일 저장 완료: {SUBMISSION_PATH}")

📊 예측용 연도별/여름 통계 사전 계산 중...
✅ 예측 데이터 생성 및 추론 시작


100%|██████████| 1893/1893 [00:00<00:00, 16752.65it/s]


✅ 제출 파일 저장 완료: ./submission/BEST_oversampling_X_20251127_222740.csv


# ==============================================